# Module 9 · 音訊 3：現代音訊表示（Whisper / wav2vec2）

> **本節定位｜2026 主流核心**
> 和文字的 `AutoTokenizer`、影像的 `AutoImageProcessor` 對應，
> 音訊有 **`AutoFeatureExtractor`**：載入哪個模型，就用它配套的前處理，
> 自動產出正確的輸入張量。

## 學習目標
- 用 `AutoFeatureExtractor` 產出 **Whisper**（log-mel `input_features`）與
  **wav2vec2**（正規化波形 `input_values`）的模型輸入。
- 用預訓練模型取得**音訊嵌入**，作為下游任務（分類/檢索）的特徵。
- 理解兩種典型輸入路線：頻譜式 (Whisper) vs 原始波形式 (wav2vec2)。

<!-- concept-image:03_modern_audio_representations_fig1 -->
<div align="center">
<img src="concept_images/03_modern_audio_representations_fig1_extractor_20260611_221852.png" alt="AutoFeatureExtractor：模型配套的自動前處理" width="640" style="max-width:100%;">
<br><sub>圖 1 · AutoFeatureExtractor：模型配套的自動前處理</sub>
</div>

In [ ]:
import numpy as np
import librosa
# 真實英語語音（librosa 內建的 LibriSpeech 片段），16kHz —— 正是 Whisper/wav2vec2 的場景
sr = 16000
wave, sr = librosa.load(librosa.example("libri1"), sr=sr, duration=5.0)
wave = wave.astype(np.float32)
print(f"真實語音 librosa 'libri1': {wave.shape}  {sr}Hz、{len(wave)/sr:.1f} 秒")

try:
    import torch
    HAS = True
except Exception:
    HAS = False
    print("未安裝 torch，請 `uv sync --extra multimodal`。")

## 1. Whisper：log-mel 頻譜輸入

Whisper（語音辨識 ASR）吃 **log-mel 頻譜**，`AutoFeatureExtractor` 會自動把任意長度
音訊補/裁成固定 30 秒並算成 80 維 mel，輸出 `input_features (B, 80, 3000)`。

<!-- concept-image:03_modern_audio_representations_fig2 -->
<div align="center">
<img src="concept_images/03_modern_audio_representations_fig2_models_20260611_221936.png" alt="Whisper vs wav2vec2：兩種輸入路線" width="640" style="max-width:100%;">
<br><sub>圖 2 · Whisper vs wav2vec2：兩種輸入路線</sub>
</div>

In [ ]:
if HAS:
    try:
        from transformers import AutoFeatureExtractor
        fe = AutoFeatureExtractor.from_pretrained("openai/whisper-tiny")
        feat = fe(wave, sampling_rate=sr, return_tensors="pt")
        print(f"Whisper input_features: {tuple(feat['input_features'].shape)}  (B, n_mels=80, T=3000)")
        print("→ 不論原音多長，都被標準化成固定形狀（補/裁到 30 秒）。")
    except Exception as e:
        print("（未能下載 Whisper 前處理設定，略過）", e)

## 2. wav2vec2：正規化原始波形輸入

wav2vec2 / HuBERT 直接吃**正規化後的原始波形**，輸出 `input_values (B, samples)`，
由模型內部的卷積前端自己學特徵。

In [ ]:
if HAS:
    try:
        from transformers import AutoFeatureExtractor
        fe2 = AutoFeatureExtractor.from_pretrained("facebook/wav2vec2-base-960h")
        feat2 = fe2(wave, sampling_rate=sr, return_tensors="pt")
        print(f"wav2vec2 input_values: {tuple(feat2['input_values'].shape)}  (B, samples)")
        print("→ 不轉頻譜，直接餵正規化波形。")
    except Exception as e:
        print("（未能下載 wav2vec2 前處理設定，略過）", e)

## 3. 取得音訊嵌入（供下游分類/檢索）

把音訊餵進預訓練模型，取 `last_hidden_state (B, T, D)`，對時間軸池化得到「一段音訊一向量」`(B, D)`。

<!-- concept-image:03_modern_audio_representations_fig3 -->
<div align="center">
<img src="concept_images/03_modern_audio_representations_fig3_embeddings_20260611_222014.png" alt="音訊嵌入提取與下游任務" width="640" style="max-width:100%;">
<br><sub>圖 3 · 音訊嵌入提取與下游任務</sub>
</div>

In [ ]:
if HAS:
    try:
        from transformers import AutoFeatureExtractor, AutoModel
        fe2 = AutoFeatureExtractor.from_pretrained("facebook/wav2vec2-base-960h")
        model = AutoModel.from_pretrained("facebook/wav2vec2-base-960h")
        model.eval()
        inp = fe2(wave, sampling_rate=sr, return_tensors="pt")
        with torch.no_grad():
            out = model(**inp)
        hidden = out.last_hidden_state            # (B, T, D)
        emb = hidden.mean(dim=1)                  # (B, D) 時間平均池化
        print(f"last_hidden_state: {tuple(hidden.shape)}  (B, T, D)")
        print(f"池化後音訊嵌入   : {tuple(emb.shape)}  (B, D) ← 可接分類器/做檢索")
    except Exception as e:
        print("（未能下載 wav2vec2 權重，略過）", e)

## 小結

| 模型 | 輸入 | 形狀 | 典型任務 |
|:--|:--|:--|:--|
| Whisper | log-mel 頻譜 `input_features` | `(B, 80, 3000)` | 語音辨識 (ASR)、翻譯 |
| wav2vec2 / HuBERT | 正規化波形 `input_values` | `(B, samples)` | 語音表示、分類、ASR |
| （取嵌入） | `last_hidden_state` 池化 | `(B, D)` | 音訊分類、檢索 |

**下一節 `04_urban_sound_case`**：把整套前處理用在環境聲音分類案例。